In [ ]:
!pip install langgraph langchain groq gradio

In [ ]:
from langgraph.graph import StateGraph, END
from typing import TypedDict
from langchain.chat_models import ChatOpenAI
from langchain_groq import ChatGroq
import gradio as gr

In [ ]:
import os

os.environ["GROQ_API_KEY"] = "YOUR_GROQ_API_KEY"

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0
)

In [ ]:
class RocketState(TypedDict):
    temperature: float
    threshold: float
    status: str
    explanation: str

In [ ]:
def safety_check(state: RocketState):
    temp = state["temperature"]
    threshold = state["threshold"]

    if temp > threshold:
        status = "🚨 SHUTDOWN INITIATED"
    else:
        status = "✅ ENGINE STABLE"

    return {
        "status": status,
        "temperature": temp,
        "threshold": threshold
    }

In [ ]:
def generate_explanation(state: RocketState):
    prompt = f"""
    You are an aerospace safety AI.

    Analyze the rocket engine condition:
    - Temperature: {state['temperature']} °C
    - Threshold: {state['threshold']} °C
    - Status: {state['status']}

    Provide a professional diagnostic explanation.
    """

    response = llm.invoke(prompt)

    return {
        "explanation": response.content
    }

In [ ]:
graph = StateGraph(RocketState)

graph.add_node("safety_check", safety_check)
graph.add_node("llm_explanation", generate_explanation)

graph.set_entry_point("safety_check")

graph.add_edge("safety_check", "llm_explanation")
graph.add_edge("llm_explanation", END)

app = graph.compile()

In [ ]:
def format_output(result):
    return f"""
    🚀 **Rocket Engine Safety System**
    --------------------------------------

    🌡️ Temperature: {result['temperature']} °C  
    ⚙️ Threshold: {result['threshold']} °C  

    🧭 Status:
    {result['status']}

    📊 AI Diagnostic:
    {result['explanation']}
    """

In [ ]:
def run_agent(temp, threshold):
    result = app.invoke({
        "temperature": temp,
        "threshold": threshold
    })

    return format_output(result)

In [ ]:
with gr.Blocks(theme=gr.themes.Soft()) as demo:

    gr.HTML("""
    <div style="text-align:center;">
        <h1 style="color:#0b3d91;">🚀 Rocket Engine Safety Shutdown</h1>
        <h3 style="color:gray;">by Kumampati Aerospace</h3>
        <marquee behavior="scroll" direction="left" 
                 style="color:red; font-weight:bold;">
            Real-Time Engine Monitoring • Autonomous Safety Reflex System • AI Diagnostics Active
        </marquee>
        <hr>
    </div>
    """)

    with gr.Row():
        temperature = gr.Slider(0, 2000, value=500, label="Engine Temperature (°C)")
        threshold = gr.Slider(0, 2000, value=1000, label="Shutdown Threshold (°C)")

    run_btn = gr.Button("Run Safety Check", variant="primary")

    output = gr.Markdown()

    run_btn.click(
        fn=run_agent,
        inputs=[temperature, threshold],
        outputs=output
    )

demo.launch()